# Módulo 3 en una clase — Aprendizaje no supervisado: agrupar y resumir sin *y*

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Módulo 3 — No supervisados (Sesiones 7, 8 y 9 en resumen)

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. Explicar qué cambia cuando **no hay *y***: no hay respuesta que imitar ni accuracy que calcular.
2. Explicar qué hace por dentro cada algoritmo del módulo: **K-means** (centroides), **DBSCAN** (densidad), **PCA** (direcciones de máxima varianza) y **t-SNE** (vecindades).
3. Elegir sus perillas (`k`, `eps`, `n_components`, `perplexity`) sabiendo que el codo, la silueta y la estabilidad son **pistas**, no veredictos.
4. Reconocer las dos trampas grandes del módulo: **agrupar por unidades de medida** (no escalar) y **creerle a un dibujo** de t-SNE.
5. Hacer el flujo completo con datos de Colombia: escalar → PCA → K-means → perfil → mapa → lo que **no** se puede concluir.

> 💾 **Antes de ejecutar nada:** menú *Archivo → Guardar una copia en Drive* y trabaje en esa copia. Si Colab avisa que el notebook no es de Google, elija *Ejecutar de todos modos*.

## Dónde estamos en el pipeline

```
 ┌────────┐   ┌──────────────────┐   ┌────────┐   ┌────────────┐   ┌────────────────┐
 │ DATOS  │ → │ PREPROCESAMIENTO │ → │ MODELO │ → │ EVALUACIÓN │ → │ INTERPRETACIÓN │
 └────────┘   └──────────────────┘   └────────┘   └────────────┘   └────────────────┘
                 ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲     ▲▲▲▲▲▲▲▲                      ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲
   HOY: modelos SIN y. Escalar deja de ser opcional, el modelo busca estructura y la interpretación es todo.
```

🔍 **Predice** · 👀 **Mire** · ✏️ **Código clave** (ya resuelto: léalo antes de ejecutarlo) · 💥 **Trampa** · 📘 **Para entender** (lectura en casa) · ⚡ **Mini-reto**.

> **Hoy es un resumen del Módulo 3.** Este notebook va **completo**: ejecute de arriba abajo, lea cada celda y haga las predicciones antes de mirar. Los notebooks completos de cada sesión siguen en el repositorio para profundizar: [Sesión 7 · K-means y DBSCAN](https://colab.research.google.com/github/Nitobest/curso-aad-python/blob/main/sesiones/S07/S07_estudiante.ipynb) · [Sesión 8 · PCA y t-SNE](https://colab.research.google.com/github/Nitobest/curso-aad-python/blob/main/sesiones/S08/S08_estudiante.ipynb) · [Sesión 9 · Municipios de Colombia](https://colab.research.google.com/github/Nitobest/curso-aad-python/blob/main/sesiones/S09/S09_estudiante.ipynb).
>
> **Datos:** datos de laboratorio (`make_blobs`, `make_moons`), **Palmer Penguins** (344 pingüinos, CC0), los **dígitos 8×8** de scikit-learn y el **IPM municipal del Censo 2018** (DANE).

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

pd.set_option("display.max_columns", 40)
REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVOS = {"penguins.csv": "datos/penguins.csv", "municipios_ipm_2018.csv": "datos/municipios_ipm_2018.csv",
            "figuras.py": "tools/figuras.py"}

def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo}. Use el Plan B (celda siguiente).")
    return False

for archivo, ruta in ARCHIVOS.items():
    asegurar_datos(archivo, REPO + ruta)
try:
    import figuras as fg
except ModuleNotFoundError:   # Plan B: GitHub no respondió y no llegó figuras.py
    from google.colab import files
    print("⚠️ No llegó figuras.py: súbalo (el profesor lo comparte por el canal del curso).")
    files.upload()
    import figuras as fg
print("Listo:", ", ".join(ARCHIVOS))

<details><summary><b>Plan B</b> — si la celda anterior avisó que no pudo bajar algún archivo</summary>

La celda siguiente revisa sola qué falta y solo en ese caso le pide subirlo.
</details>

In [ ]:
# Plan B: solo actúa si falta algún archivo. Con "Ejecutar todo" no se detiene.
from pathlib import Path
faltan = [a for a in ARCHIVOS if not Path(a).exists()]
if not faltan:
    print("✓ Todo disponible: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ Faltan:", faltan, "→ súbalos")
    files.upload()

In [ ]:
fg.pipeline(["preprocesamiento", "modelo", "interpretación"], modulo=3, subtitulo="HOY: el Módulo 3 en una clase — agrupar y resumir sin y");

## 1. Se acabó la *y*

En los módulos 1 y 2 cada fila traía su respuesta (¿hubo víctimas?, ¿cuánto cobra el seguro?, ¿abrió el depósito?) y el modelo aprendía a imitarla. En el **aprendizaje no supervisado** no hay respuesta: solo las *X*. La pregunta cambia de *"¿qué va a pasar?"* a *"¿qué estructura tienen estos datos?"*.

In [ ]:
fg.supervisado_vs_no_supervisado();

### El mapa de hoy: dos tareas, cuatro algoritmos

| Tarea | Pregunta | Algoritmo | Idea en una frase | Sección |
|---|---|---|---|---|
| **Agrupar** (clustering) | ¿qué grupos de filas parecidas hay? | **K-means** | cada grupo es una nube alrededor de un centro (centroide) | 2, 3, 5 |
| | | **DBSCAN** | un grupo es una zona densa, tenga la forma que tenga; lo aislado es ruido | 4 |
| **Resumir** (reducción de dimensionalidad) | ¿cómo veo o comprimo muchas columnas en pocas? | **PCA** | ejes nuevos en la dirección donde más se esparcen los datos | 6 |
| | | **t-SNE** | un dibujo 2D donde cada punto conserva a sus vecinos | 7 |

Y una regla que vale para los cuatro: **miden distancias**, así que las unidades de medida importan (sección 5). Al final (sección 8) los usamos juntos con datos de Colombia y en la sección 9 queda **una ficha por algoritmo** para repasar.

**Clustering** (agrupamiento) es la tarea no supervisada más común: partir las filas en grupos de filas **parecidas entre sí** y **distintas de las de otros grupos**. Se usa para segmentar clientes, agrupar municipios con realidades parecidas, encontrar comportamientos raros (fraude, sensores dañados) o simplemente para entender un dataset nuevo.

Empezamos con datos de **laboratorio**: `make_blobs` fabrica nubes de puntos alrededor de centros que nosotros elegimos. Nosotros sabemos la verdad; el algoritmo no la va a ver.

### 📘 Para entender — el aprendizaje no supervisado en la práctica

**Qué es.** Encontrar estructura en datos **sin una respuesta correcta**. No hay *y* que imitar ni accuracy que calcular: el modelo propone una organización de los datos y el analista decide si sirve.

**Las cuatro tareas más comunes:**

| Tarea | Pregunta | Ejemplos reales | En el curso |
|---|---|---|---|
| **Clustering** (agrupar) | ¿Qué grupos de filas parecidas hay? | segmentar clientes de un banco; agrupar tiendas por su patrón de ventas; tipologías de municipios | K-means, DBSCAN (secciones 2–5); municipios (sección 8) |
| **Reducción de dimensionalidad** | ¿Cómo resumo muchas columnas en pocas? | ver un dataset de 50 variables en un plano; comprimir antes de otro modelo | PCA, t-SNE (secciones 6–7) |
| **Detección de anomalías** | ¿Qué filas son raras? | transacciones sospechosas, sensores dañados, facturas atípicas | el "ruido" de DBSCAN (sección 4) |
| **Reglas de asociación** | ¿Qué cosas aparecen juntas? | "quien compra pan compra leche" (canasta de mercado) | — (no es tema del curso) |

**El flujo de trabajo de un clustering**, que va a repetir en la sección 8 y, si quiere, en su proyecto:

1. **Elegir las columnas** que describen lo que quiere agrupar (y dejar por fuera identificadores y etiquetas).
2. **Escalar** (casi siempre: estos algoritmos miden distancias).
3. **Ajustar** el algoritmo y elegir sus perillas (`k`, `eps`).
4. **Mirar** los grupos: tamaños, perfil de cada uno en unidades originales.
5. **Validar** con algo que el algoritmo no vio (una etiqueta conocida, un mapa, el criterio de un experto).
6. **Nombrar** los grupos y decir qué **no** se puede concluir.

> ⚠️ Un algoritmo de clustering **siempre** devuelve grupos, aunque los datos no tengan ninguno. Que existan grupos en la salida no prueba que existan en la realidad.

In [ ]:
from sklearn.datasets import make_blobs

X_lab, grupo_verdadero = make_blobs(n_samples=500, centers=[(-5, -2), (-1, 5), (2, -1), (4.5, 1.5)],
                                    cluster_std=[1.2, 1.3, 1.0, 1.0], random_state=7)
X_lab = pd.DataFrame(X_lab, columns=["x1", "x2"])
X_lab.head()

### 🔍 Predice
Mire la figura de abajo **sin** pensar en cuántos centros usamos. ¿Cuántos grupos ve usted?

**Su predicción:** ____ grupos

In [ ]:
fg.puntos_sin_color(X_lab);

**Fabricamos 4**, pero mucha gente ve **3**: los dos de la derecha casi se tocan. Guarde esa duda; va a volver cuando el algoritmo tenga que elegir.

## 2. K-means: asignar y promediar

K-means necesita que le digamos **cuántos grupos** buscar (`k`). Después repite dos pasos:

1. **Asignar:** cada punto se va con el **centroide** (el centro del grupo) más cercano.
2. **Promediar:** cada centroide se muda al **promedio** de los puntos que le tocaron.

…hasta que los centroides ya no se mueven. Ese vaivén de asignar y promediar se llama **algoritmo de Lloyd**: es el K-means básico. Los primeros centroides se escogen al azar (en la versión básica; scikit-learn usa por defecto **k-means++**, que los escoge separados entre sí).

> 🧪 **Laboratorio** (para jugar después de clase, o ya si va adelantado): [**Laboratorio 7 · K-means**](https://nitobest.github.io/curso-aad-python/laboratorio/07_kmeans.html) · asignar y mover, paso a paso; el reto de los pingüinos con y sin escalar. Tres pestañas: **Mira**, **Juega** y **Reto** (prediga antes de correr). Funciona en el celular.

In [ ]:
fg.kmeans_iteraciones(X_lab, k=4, semilla=8, mostrar=(0, 1, 2, 4, -1));

### 📐 Fundamento
Lo que K-means intenta hacer pequeño es la **inercia**: la suma, sobre todos los puntos, de la distancia al cuadrado a su centroide.

$$\text{inercia} = \sum_{i=1}^{n} \lVert x_i - \mu_{c(i)} \rVert^2$$

- $x_i$ es el punto $i$; $\mu_{c(i)}$ es el centroide del grupo al que quedó asignado.
- Inercia **baja** = puntos apretados alrededor de su centro.

Es la misma idea de los **residuales al cuadrado** de la regresión (Sesión 4): allí cada punto se comparaba con la recta; aquí, con su centroide.

<details><summary><b>Para quien quiera más:</b> por qué la inercia nunca sube, y por qué igual puede quedar mal</summary>

El paso *asignar* no puede subir la inercia (cada punto se va al centro más cercano, o se queda). El paso *promediar* tampoco: el promedio es justamente el punto que minimiza la suma de distancias² a un grupo de puntos. Por eso el algoritmo siempre termina. Pero termina en un **mínimo local**: el mejor arreglo *cerca* de donde arrancó, no necesariamente el mejor posible. Encontrar el mínimo global es un problema computacionalmente muy difícil (NP-difícil), así que en la práctica se arranca varias veces (`n_init`) y se escogen arranques inteligentes (**k-means++**: el primer centroide al azar y cada siguiente lejos de los anteriores).
</details>

![Dentro del .fit() — K-means y DBSCAN](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S07/img/dentro_del_fit_kmeans_dbscan.png)

### ✏️ Código clave — entrene K-means
*Ya está resuelto en la celda siguiente: lea qué hace antes de ejecutarlo.*

Construya un `KMeans(n_clusters, n_init, random_state)` con `k = 4` grupos, 10 arranques (`n_init`) y `random_state=42`, y entrénelo con `X_lab`. Guárdelo en `kmeans`. Ojo: **no hay `y`** en el `.fit()`.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_lab)
print("grupo de las primeras 10 filas:", kmeans.labels_[:10])
print("inercia:", round(kmeans.inertia_, 1))
pd.DataFrame(kmeans.cluster_centers_, columns=["x1", "x2"]).round(2)

### 📘 Para entender — K-means, para qué sirve y cómo se configura

**Qué es.** Un algoritmo que parte los datos en $k$ grupos, cada uno representado por su **centroide** (el promedio de sus puntos), buscando que los puntos queden lo más cerca posible de su centroide (inercia mínima).

**Las perillas:**

| Hiperparámetro | Qué controla | Recomendación |
|---|---|---|
| `n_clusters` ($k$) | cuántos grupos | lo decide usted, con codo, silueta y, sobre todo, la utilidad |
| `n_init` | cuántas veces arranca desde distintos puntos | 10 o más; se queda con el de menor inercia |
| `init` | cómo elige los primeros centroides | `"k-means++"` (por defecto): separados entre sí |
| `random_state` | la semilla | fíjela para poder repetir el resultado |

**Lo que K-means supone sin decirlo:**

- Los grupos son **nubes redondas** alrededor de un centro (no lunas, no anillos, no franjas).
- Los grupos tienen **tamaños y dispersiones parecidos**.
- Todas las variables están en **escalas comparables** (escale siempre).
- No hay ruido: **cada** punto queda en algún grupo, incluso los raros.

**Dónde se usa:**

- **Segmentación de clientes**: por ejemplo, con *recencia, frecuencia y monto* de compra (análisis RFM) se arman grupos como "clientes fieles", "en riesgo" o "nuevos".
- **Ubicación de centros de distribución**: los centroides de las coordenadas de los clientes son un primer candidato de dónde poner bodegas.
- **Comprimir colores** de una imagen (cada píxel toma el color de su centroide).
- Como paso previo para resumir muchas filas en $k$ "prototipos".

| Ventajas | Desventajas |
|---|---|
| rápido, incluso con millones de filas (`MiniBatchKMeans`) | hay que dar $k$ |
| centroides fáciles de describir y `.predict()` para filas nuevas | solo encuentra grupos redondos; sensible a valores extremos |
| fácil de explicar | depende de la semilla y del escalado |

`labels_` es el grupo de cada fila (0, 1, 2, 3: los números son **nombres**, no un orden) y `cluster_centers_` son los centroides. Eso es **todo** lo que aprende K-means: k puntos.

In [ ]:
fg.kmeans_resultado(X_lab, kmeans);

Como aprendió centroides, K-means puede ubicar filas **nuevas**: `.predict()` mide la distancia a cada centroide y escoge la más corta.

In [ ]:
nuevos = pd.DataFrame({"x1": [0, 4], "x2": [6, 0]})
nuevos.assign(grupo=kmeans.predict(nuevos))

## 3. ¿Cuántos grupos? La perilla `k`

### 🔍 Predice
Si le pedimos a K-means **6** grupos sobre los mismos datos, ¿qué cree que hace? ¿Se niega, deja grupos vacíos, o…?

**Su predicción:** ____

In [ ]:
fg.kmeans_k_efecto(X_lab, ks=(2, 3, 4, 6));

**Entrega exactamente 6.** K-means nunca dice "aquí no hay 6 grupos": parte nubes reales en pedazos. La responsabilidad de escoger `k` es nuestra. Dos ayudas:

### Dos ayudas para elegir `k`: el codo y la silueta

- **Inercia (codo):** la inercia siempre baja al subir `k` (con un grupo por punto vale 0). Se busca el **codo**: el `k` donde agregar un grupo deja de ayudar mucho.
- **Silueta:** para cada punto, $a$ = distancia media a los puntos de **su** grupo y $b$ = distancia media al **grupo vecino más cercano**. Su silueta es $s = (b - a) / \max(a, b)$: cerca de **1** = bien ubicado, **0** = en la frontera, **negativo** = probablemente mal asignado. La figura muestra el promedio de todos los puntos.

![Cómo medir clusters sin y](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S07/img/medir_sin_y.png)

In [ ]:
fg.codo_silueta(X_lab, ks=range(1, 8), marcar=4);

### 📘 Para entender — cómo evaluar un clustering sin *y*

**Métricas internas** (solo miran la geometría de los datos):

| Métrica | Qué mide | Cómo se lee |
|---|---|---|
| **Inercia** | suma de distancias² de cada punto a su centroide | siempre baja al subir $k$: se busca el **codo** |
| **Silueta** | qué tan cerca está cada punto de su grupo comparado con el grupo vecino | de −1 a 1; más alto = grupos más separados |
| **Davies-Bouldin** | qué tan parecido es cada grupo a su vecino más parecido | más **bajo** es mejor |
| **Calinski-Harabasz** | dispersión entre grupos frente a dispersión dentro de los grupos | más alto es mejor |

La silueta de un punto, con un ejemplo: si su distancia media a los de **su** grupo es $a = 1$ y a los del grupo vecino es $b = 4$, entonces

$$s = \frac{b - a}{\max(a, b)} = \frac{4 - 1}{4} = 0{,}75 \quad \text{(bien ubicado)}$$

Si $a = b$, $s = 0$ (está en la frontera); si $a > b$, $s < 0$ (está más cerca del otro grupo).

**Validación externa** (más valiosa): comparar los grupos con algo que el algoritmo **no** usó, como la especie de los pingüinos (sección 5), un mapa (sección 8) o el juicio de alguien que conoce el negocio.

**Validación por utilidad**: ¿los grupos sirven para tomar una decisión distinta en cada uno? Cuatro segmentos de clientes que reciben la misma campaña no sirven de nada, aunque su silueta sea alta.

**Lectura:** la inercia cae fuerte hasta `k = 3–4` y después casi no mejora (el **codo**: 2 027 con k = 3, 1 157 con k = 4, 1 033 con k = 5). La **silueta** es máxima en `k = 3` (0,63), no en 4 (0,61): los dos grupos de la derecha están tan cerca que, geométricamente, separarlos "cuesta". La verdad de laboratorio era 4. Ninguna métrica decide sola: son pistas para una decisión que también depende de para qué quiere los grupos.

### La semilla importa
¿Qué pasa si K-means arranca **una sola vez** (`n_init=1`) desde centroides tomados al azar (`init='random'`, la versión básica)? Mismo `k = 4`, cuatro semillas distintas:

In [ ]:
fg.kmeans_semillas(X_lab, k=4, semillas=(0, 1, 2, 3));

Con las semillas 0 y 2, K-means se queda atascado en un **mínimo local**: dos centroides se pelean una nube y otra nube queda partida. La inercia lo delata (1 915 y 1 893 contra 1 157). `n_init=10` corre diez arranques y se queda con el de menor inercia; por eso lo usamos siempre. Con el arranque por defecto de scikit-learn (k-means++), esas mismas cuatro semillas llegan a 1 157 aun con `n_init=1`. Guarde esta idea para la sección 7: **un resultado que cambia con la semilla no es un hallazgo.**

## 4. Cuando los grupos no son redondos: DBSCAN

K-means supone que cada grupo es una nube alrededor de un centro. ¿Y si los grupos tienen otra forma? `make_moons` fabrica dos lunas entrelazadas; les agregamos 25 puntos sueltos (ruido).

In [ ]:
from sklearn.datasets import make_moons

lunas, _ = make_moons(n_samples=400, noise=0.07, random_state=0)
sueltos = np.random.default_rng(1).uniform([-1.3, -0.8], [2.3, 1.3], size=(25, 2))
X_lunas = pd.DataFrame(np.vstack([lunas, sueltos]), columns=["x1", "x2"])
fg.kmeans_vs_dbscan(X_lunas, k=2, eps=0.15);

K-means parte con una recta, porque solo sabe de centros. **DBSCAN** (*Density-Based Spatial Clustering of Applications with Noise*) piensa distinto: un grupo es una **zona densa** de puntos, tenga la forma que tenga.

In [ ]:
fg.dbscan_idea(X_lunas, eps=0.15, min_samples=5);

### 📐 Fundamento
DBSCAN tiene dos perillas: un radio $\varepsilon$ (`eps`) y un mínimo de vecinos (`min_samples`).

- **Núcleo:** punto con al menos `min_samples` puntos (contándose a sí mismo) a distancia $\le \varepsilon$.
- **Borde:** punto que no es núcleo pero está a $\le \varepsilon$ de algún núcleo.
- **Ruido:** todo lo demás. DBSCAN le pone la etiqueta **−1**.

Un cluster es un conjunto de núcleos **encadenados** (cada uno a $\le \varepsilon$ del siguiente) más sus bordes. No hay $k$ ni función de pérdida: el número de grupos **sale** de los datos.

### ✏️ Código clave — DBSCAN sobre las lunas
*Ya está resuelto en la celda siguiente: lea qué hace antes de ejecutarlo.*

Construya un `DBSCAN(eps, min_samples)` con radio 0,15 y 5 vecinos mínimos, y guarde en `etiquetas_db` las etiquetas de `X_lunas`. DBSCAN no tiene `.predict()`; se usa `.fit_predict()`, que entrena y devuelve las etiquetas de las mismas filas.

In [ ]:
from sklearn.cluster import DBSCAN

etiquetas_db = DBSCAN(eps=0.15, min_samples=5).fit_predict(X_lunas)
pd.Series(etiquetas_db).value_counts().rename("puntos").rename_axis("etiqueta (−1 = ruido)")

### 📘 Para entender — DBSCAN, para qué sirve y cómo elegir `eps`

**Qué es.** Un algoritmo que define un grupo como una **zona densa** de puntos: puntos con muchos vecinos cerca se encadenan entre sí, y los que quedan aislados se marcan como **ruido** (etiqueta −1). No hay que decirle cuántos grupos buscar.

**Las perillas:**

| Hiperparámetro | Qué controla | Efecto |
|---|---|---|
| `eps` ($\varepsilon$) | el radio de "vecindad": qué tan cerca es cerca | grande = todo se une en un grupo; pequeño = casi todo es ruido |
| `min_samples` | cuántos vecinos necesita un punto para ser núcleo | grande = grupos más exigentes y más ruido; regla práctica: el doble del número de variables |

**Cómo elegir `eps` sin adivinar:** calcule, para cada punto, la distancia a su vecino número `min_samples`, ordénelas y grafíquelas. La curva sube suave y luego se dispara: el `eps` está en ese **codo**.

```python
from sklearn.neighbors import NearestNeighbors
d, _ = NearestNeighbors(n_neighbors=5).fit(X_lunas).kneighbors(X_lunas)
plt.plot(np.sort(d[:, -1])); plt.ylabel("distancia al 5.º vecino")
```

**Dónde se usa:**

- **Puntos calientes en mapas**: con las coordenadas de los incidentes viales de Medellín, DBSCAN encontraría las intersecciones donde se concentran los accidentes, de cualquier forma, y dejaría como ruido los incidentes aislados.
- **Detección de anomalías**: lo que queda como ruido es, por definición, lo que no se parece a nada.
- **Trayectorias de GPS**: lugares donde un vehículo o una persona se detiene con frecuencia.

| Ventajas | Desventajas |
|---|---|
| encuentra grupos de cualquier forma | un solo `eps` para todo: le cuestan grupos de densidades muy distintas |
| no hay que dar el número de grupos | muy sensible a la escala (escalar siempre) |
| separa el ruido en vez de forzarlo en un grupo | no tiene `.predict()` para filas nuevas |

### 🔍 Predice
Con `eps = 0.3` (el doble), ¿cuántos grupos encontrará? ¿Y con `eps = 0.05`?

**Su predicción:** con 0,3 → ____ · con 0,05 → ____

In [ ]:
fg.dbscan_eps(X_lunas, epsilons=(0.05, 0.1, 0.15, 0.3));

**Con 0,3, un solo grupo**: el radio es tan grande que las dos lunas se encadenan. **Con 0,05, una nube de grupitos y casi todo es ruido**: nadie tiene vecinos tan cerca. `eps` es "qué tan cerca es cerca", y eso depende de las **unidades** de las variables. Si una variable viene en miles y otra en décimas, el mismo círculo es enorme en una dirección y diminuto en la otra. Esa frase es la clave de la siguiente sección.

![¿K-means o DBSCAN?](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S07/img/kmeans_o_dbscan.png)

### 📘 Otros algoritmos de clustering que va a encontrar

| Algoritmo | Idea | Úselo cuando… | En scikit-learn |
|---|---|---|---|
| **Jerárquico (aglomerativo)** | empieza con cada punto solo y va uniendo los dos grupos más parecidos; el resultado es un árbol (**dendrograma**) que se corta a la altura que se quiera | pocos datos (miles de filas) y quiere ver la estructura a varios niveles | `AgglomerativeClustering`; dendrograma con `scipy` |
| **Mezcla de gaussianas** | supone que cada grupo es una "campana" (puede ser alargada) y da a cada punto una **probabilidad** de pertenecer a cada grupo | grupos elípticos o que se traslapan; quiere probabilidades, no etiquetas duras | `GaussianMixture` |
| **HDBSCAN** | DBSCAN que se adapta a grupos de distinta densidad, sin elegir `eps` | como DBSCAN, pero con densidades muy distintas | `HDBSCAN` |

Regla práctica para empezar: K-means si espera grupos compactos y quiere describirlos con un centro; DBSCAN o HDBSCAN si espera formas raras o ruido; jerárquico si quiere ver cómo se van uniendo los grupos.

## 5. Datos reales: 342 pingüinos

En la Estación Palmer (Antártida) midieron pingüinos de **tres especies**: **Adelie** (pingüino de Adelia), **Chinstrap** (barbijo) y **Gentoo** (papúa). Desde aquí usamos los nombres en inglés, que son los que trae la tabla y salen en las figuras. Tenemos cuatro medidas por pingüino: largo y alto del pico (mm), largo de la aleta (mm) y masa corporal (g).

Vamos a **esconder la especie**: agrupamos solo con las cuatro medidas y al final miramos si los grupos coinciden con las especies. La especie es nuestra "verdad de laboratorio" en datos reales.

In [ ]:
pinguinos = pd.read_csv("penguins.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
pinguinos = pinguinos.dropna(subset=medidas).reset_index(drop=True)
print(pinguinos.shape)
pinguinos[medidas].describe().round(1)

### 🔍 Predice
Le pedimos a K-means `k = 3` grupos usando las cuatro medidas **tal como vienen**. Hay tres especies. ¿Qué porcentaje de pingüinos cree que quedará en el grupo de "su" especie?

**Su predicción:** ____ %

### ✏️ Código clave — K-means con las medidas tal como vienen
*Ya está resuelto en la celda siguiente: lea qué hace antes de ejecutarlo.*

El mismo `KMeans(n_clusters, n_init, random_state)` de antes, ahora con **3** grupos, entrenado con `pinguinos[medidas]` tal como vienen. Guárdelo en `km_crudo`.

In [ ]:
km_crudo = KMeans(n_clusters=3, n_init=10, random_state=42).fit(pinguinos[medidas])
fg.tabla_cruzada(pinguinos["species"], km_crudo.labels_);

**68 %.** Gentoo queda partido en dos grupos y Adelie y Chinstrap quedan revueltos. ¿Las medidas no sirven para distinguir especies? Miremos cómo partió K-means a los pingüinos:

In [ ]:
pinguinos.assign(cluster=km_crudo.labels_).groupby("cluster")["body_mass_g"].agg(["min", "max", "mean"]).round(0)

### 💥 Agrupó por gramos

Los tres grupos son **tres franjas de peso**: menos de 3 975 g, de 4 000 a 4 950 g, y más de 4 975 g. Las otras tres medidas casi no participaron.

<details><summary><b>💥 ¿Por qué pasó esto?</b> (ábralo después de la discusión)</summary>

K-means mide **distancias**. La masa viene en gramos, con desviación estándar de **802**; el alto del pico viene en milímetros, con desviación de **2**. Dos pingüinos que difieren en 500 g y en 5 mm de pico están a una distancia de $\sqrt{500^2 + 5^2} \approx 500{,}02$: el pico no aporta **nada**. K-means no agrupó por especie, agrupó por la **unidad de medida** más grande.

Es el arco de escalado que abrimos en la Sesión 2 y vimos morder a k-NN en la Sesión 5: todo algoritmo que mide distancias necesita variables en escalas comparables. Si hubiéramos pesado a los pingüinos en kilos, el resultado habría sido otro: **una decisión de unidades cambió los "hallazgos"**.
</details>

### ✏️ Código clave — escale y repita
*Ya está resuelto en la celda siguiente: lea qué hace antes de ejecutarlo.*

`StandardScaler` (Sesión 2) deja cada medida con promedio 0 y desviación 1. No hay *y* ni prueba aquí: se ajusta con todos los pingüinos. Entrene el **mismo** K-means de la celda anterior, ahora con `Z`, y guárdelo en `km_z`.

In [ ]:
from sklearn.preprocessing import StandardScaler

escalador = StandardScaler()
Z = escalador.fit_transform(pinguinos[medidas])
km_z = KMeans(n_clusters=3, n_init=10, random_state=42).fit(Z)
fg.tabla_cruzada(pinguinos["species"], km_z.labels_);

In [ ]:
fg.clusters_por_unidad(pinguinos, "body_mass_g", "bill_length_mm", km_crudo.labels_, km_z.labels_);

**De 68 % a 92 %**, con los mismos datos y el mismo algoritmo. Gentoo queda completo en un grupo; el error que queda está entre Adelie y Chinstrap, que tienen cuerpos muy parecidos y se distinguen sobre todo por el pico.

### ¿Cómo es cada grupo?
Los centroides de `km_z` están en unidades *z* (promedio 0, desviación 1). `inverse_transform` los devuelve a milímetros y gramos para **describir** cada grupo con palabras: ese es el paso que convierte etiquetas (0, 1, 2) en hallazgos.

In [ ]:
perfil = pd.DataFrame(escalador.inverse_transform(km_z.cluster_centers_), columns=medidas).round(1)
perfil["pingüinos"] = np.bincount(km_z.labels_)
perfil["especie más común"] = pinguinos.groupby(km_z.labels_)["species"].agg(lambda s: s.mode()[0])
perfil

**Lectura:** un grupo de pingüinos **grandes** (aleta de 217 mm, 5 076 g y pico delgado de 15 mm: casi todos Gentoo), uno de cuerpo pequeño con pico **largo y alto** (47,5 × 18,8 mm: Chinstrap) y uno de pico **corto** (38,2 mm: Adelie). Los nombres salen del perfil, no del algoritmo: la especie solo se usó al final, para comprobar.

## 6. PCA: resumir muchas columnas en pocas

Hasta aquí dibujamos a los pingüinos con 2 de sus 4 medidas. Para verlas todas hay que mirar **todas las parejas**:

In [ ]:
fg.todas_las_parejas(pinguinos, medidas, color_col="species");

Seis gráficas para 4 variables ya es incómodo; con los 64 píxeles de una imagen serían 2 016. La **reducción de dimensionalidad** busca pocas columnas nuevas que resuman muchas, perdiendo lo menos posible.

### 📘 Para entender — ¿para qué reducir dimensiones?

**Qué es.** Reemplazar muchas columnas por unas pocas columnas nuevas que conserven lo más importante de la información.

**Para qué se usa:**

| Uso | Ejemplo |
|---|---|
| **Ver** datos de muchas variables en un plano | ubicar 1 122 municipios según 15 privaciones en una sola gráfica (sección 8) |
| **Comprimir** antes de otro modelo | pasar de 64 píxeles a 21 componentes antes de clasificar |
| **Quitar redundancia** | variables que dicen casi lo mismo (ingreso, gasto, estrato) se resumen en un solo eje |
| **Construir índices** | un "índice socioeconómico" o un "índice de tamaño" a partir de varias medidas |
| **Quitar ruido** | quedarse con los componentes grandes y descartar los que solo tienen ruido |

**Dos familias:**

- **Proyecciones lineales** (PCA): cada columna nueva es una suma ponderada de las originales. Se puede interpretar y se puede aplicar a filas nuevas.
- **Métodos de vecindad** (t-SNE, UMAP): acomodan los puntos para que los vecinos sigan siendo vecinos. Dibujan mejor estructuras curvas, pero los ejes no significan nada.

**No confunda:** *seleccionar* variables (quedarse con algunas de las originales, como hace Lasso en la Sesión 4) no es lo mismo que *extraer* variables nuevas (PCA crea columnas que mezclan todas).

### PCA: la dirección en la que más se esparcen los datos

**PCA** (*Principal Component Analysis*, análisis de componentes principales) busca la dirección en la que los puntos están **más esparcidos** y la llama **PC1** (primer componente principal). Después busca la siguiente, perpendicular a la primera (**PC2**), y así. Proyectar sobre PC1 es como mirar la "sombra" de los datos sobre esa línea:

> 🧪 **Laboratorio** (para jugar después de clase, o ya si va adelantado): [**Laboratorio 8 · PCA y t-SNE**](https://nitobest.github.io/curso-aad-python/laboratorio/08_pca.html) · gire la línea hasta la dirección de máxima varianza, compare PCA y t-SNE con los pingüinos y el reto de los grupos que no existen. Tres pestañas: **Mira**, **Juega** y **Reto** (prediga antes de correr). Funciona en el celular.

In [ ]:
fg.pca_proyeccion();

### 📐 Fundamento
Cada componente es una **suma ponderada** de las variables (escaladas):

$$\text{PC1} = w_1 x_1 + w_2 x_2 + \dots + w_p x_p$$

Los pesos $w$ se eligen para que PC1 tenga la **máxima varianza** posible (con $\sum w_j^2 = 1$, para que no "haga trampa" agrandando los pesos). PC2 hace lo mismo con la condición de ser perpendicular a PC1 (no repetir la varianza que PC1 ya recogió), y así sucesivamente.

La **varianza explicada** del componente $k$ es la fracción de la variación total que recoge:

$$\text{varianza explicada}_k = \frac{\lambda_k}{\lambda_1 + \lambda_2 + \dots + \lambda_p}$$

donde $\lambda_k$ es la varianza de los datos sobre el componente $k$. Si PC1 y PC2 suman, digamos, 90 %, el dibujo en 2D es casi fiel.

<details><summary><b>Para quien quiera más:</b> de dónde salen los pesos</summary>

Los pesos de cada componente son los **vectores propios** de la matriz de covarianza de los datos (o de correlación, si se escaló), y las $\lambda_k$ son sus **valores propios**. No hay iteraciones ni semilla: es álgebra lineal, siempre da lo mismo (salvo el signo de cada componente, que es arbitrario). Por eso PCA es una referencia confiable para comparar con métodos más vistosos.
</details>

![Dentro del .fit() — PCA y t-SNE](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S08/img/dentro_del_fit_pca_tsne.png)

### 📘 Para entender — PCA, para qué sirve y cuántos componentes guardar

**Qué es.** Un método que rota los ejes de los datos para que el primer eje nuevo (PC1) recoja la mayor variación posible, el segundo (PC2) la mayor variación que queda siendo perpendicular al primero, y así. Cada componente es una **suma ponderada** de las variables originales.

**¿Cuántos componentes guardar?** Tres criterios habituales:

1. **Varianza acumulada**: los necesarios para llegar a 80–95 % (para comprimir).
2. **El codo** de la gráfica de varianza explicada: donde agregar otro aporta poco.
3. **Para dibujar**: 2 (o 3), sabiendo cuánta varianza se está dejando por fuera.

En código, scikit-learn elige solo si se le da una fracción:

```python
pca90 = PCA(n_components=0.90).fit(X_dig)     # los componentes necesarios para el 90 %
pca90.n_components_                           # → 21
X_comprimido = pca90.transform(X_dig)         # 64 columnas → 21
X_aprox = pca90.inverse_transform(X_comprimido)  # de vuelta a 64 (con algo de pérdida)
```

**Dónde se usa:**

- **Índices compuestos**: resumir varias condiciones de vida en un índice; PC1 suele ser "el nivel general" (lo verá en la sección 8 con la pobreza municipal).
- **Antes de una regresión** con variables muy correlacionadas: los componentes no están correlacionados entre sí.
- **Finanzas**: los rendimientos de muchas acciones se mueven juntos; PC1 suele ser "el mercado".
- **Imágenes y señales**: compresión y eliminación de ruido.
- **Control de calidad**: una fila que se reconstruye mal con pocos componentes (`inverse_transform`) es rara.

| Ventajas | Desventajas |
|---|---|
| rápido, sin semilla, siempre da lo mismo | solo capta relaciones **lineales** |
| ejes interpretables con los pesos (biplot) | máxima varianza no siempre es lo importante para su pregunta |
| `.transform()` para filas nuevas | sensible a la escala (escalar si las unidades difieren) y a valores extremos |

### PCA con los pingüinos

### 👀 Mire
Antes de escalar, la lección de la sección 5. Si hacemos PCA con las 4 medidas **tal como vienen** (gramos y milímetros), ¿qué porcentaje de la varianza recogerá PC1? Piénselo un segundo y mire.

In [ ]:
from sklearn.decomposition import PCA

pca_crudo = PCA().fit(pinguinos[medidas])
print("varianza explicada sin escalar:", [f"{v:.2%}" for v in pca_crudo.explained_variance_ratio_])
print("pesos de PC1:", {m: round(float(w), 3) for m, w in zip(medidas, pca_crudo.components_[0])})

**99,99 %**, y PC1 es casi exactamente `body_mass_g`. No es un hallazgo: la masa tiene una varianza de unos 640 000 g² y el pico, de 4 mm². PCA busca la dirección de más varianza **en las unidades que le den**. Mismo error que el K-means de la sección 5: **antes de PCA se escala** (cuando las variables vienen en unidades distintas).

### 🔍 Predice
Con las medidas **escaladas**, ¿qué porcentaje de la varianza recogerán **juntos** PC1 y PC2?

**Su predicción:** ____ %

### ✏️ Código clave — escale y ajuste PCA
*Ya está resuelto en la celda siguiente: lea qué hace antes de ejecutarlo.*

Guarde en `Z` las 4 medidas escaladas con `StandardScaler().fit_transform(...)` y en `pca` un `PCA()` ajustado con `Z` (sin `n_components`: así calcula los 4 y podemos ver todos).

In [ ]:
from sklearn.preprocessing import StandardScaler

Z = StandardScaler().fit_transform(pinguinos[medidas])
pca = PCA().fit(Z)
fg.pca_varianza(pca);

**88 %** entre los dos (69 % + 19 %). Pasar de 4 columnas a 2 conserva casi toda la varianza: el dibujo en PC1–PC2 es confiable.

### ¿Qué significa cada componente?
Los pesos están en `pca.components_` (una fila por componente). El **biplot** los dibuja como flechas encima de los pingüinos proyectados:

In [ ]:
pd.DataFrame(pca.components_[:2], columns=medidas, index=["PC1", "PC2"]).round(2)

In [ ]:
scores = pca.transform(Z)
fg.biplot(scores, pca.components_, medidas, etiquetas=pinguinos["species"], var_exp=pca.explained_variance_ratio_);

### 📘 Cómo leer un biplot, paso a paso

1. **Los puntos** son las filas (pingüinos) ubicadas según su valor en PC1 (eje horizontal) y PC2 (eje vertical). Puntos cercanos = filas parecidas en esas dos dimensiones.
2. **Las flechas** son las variables originales. Su dirección dice hacia dónde crece esa variable en el plano.
3. **Largo de la flecha**: qué tanto pesa esa variable en los dos componentes mostrados. Una flecha corta está poco representada en el dibujo.
4. **Ángulo entre flechas**: casi juntas = variables correlacionadas positivamente; opuestas = correlación negativa; en ángulo recto = poco relacionadas.
5. **Nombrar los ejes**: mire qué flechas apuntan a lo largo de cada eje. Aquí PC1 ≈ "tamaño" (aleta, masa, largo del pico) y PC2 ≈ "forma del pico".

**Dos advertencias:** el **signo** de cada componente es arbitrario (otra versión del programa podría dar el dibujo en espejo, y significa lo mismo); y el biplot solo muestra lo que cabe en PC1 y PC2 (aquí, 88 % de la varianza).

**Lectura:** PC1 suma aleta, masa y largo del pico (pesos ≈ +0,5) y resta el alto del pico (−0,4): es el **tamaño** del pingüino, con pico largo y delgado. PC2 es casi solo el pico (largo y alto): es la **forma del pico**. En ese plano los Gentoo (grandes) se van a la derecha y Adelie y Chinstrap se separan hacia arriba y abajo por el pico. Las especies no se usaron para calcular nada: solo colorean.

### Una fila entra, un número sale
El valor de PC1 de un pingüino es la suma de sus medidas escaladas por los pesos, como una predicción de la regresión de la Sesión 4:

In [ ]:
fg.pca_una_fila(Z[0], pca.components_[0], medidas, nombre="PC1", real=pinguinos.loc[0, "species"]);

## 7. Muchas dimensiones: 64 píxeles, PCA y t-SNE

Los dígitos de scikit-learn son 1 797 imágenes de 8×8 píxeles. Cada imagen es una fila de **64 columnas**.

In [ ]:
from sklearn.datasets import load_digits

X_dig, y_dig = load_digits(return_X_y=True)
print(X_dig.shape)
fg.digitos_muestra(X_dig, y_dig);

### 🔍 Predice
Con 64 columnas, ¿qué porcentaje de la varianza recogerán los **2 primeros** componentes? ¿Cuántos componentes harán falta para llegar al 90 %?

**Su predicción:** 2 componentes → ____ % · para 90 % hacen falta ____

### ✏️ Código clave — PCA sobre los dígitos
*Ya está resuelto en la celda siguiente: lea qué hace antes de ejecutarlo.*

Guarde en `pca_dig` un `PCA()` con todos los componentes, ajustado con `X_dig`. Aquí todas las columnas están en la misma unidad (intensidad de 0 a 16), así que no hace falta escalar.

In [ ]:
pca_dig = PCA().fit(X_dig)
print(f"2 componentes: {pca_dig.explained_variance_ratio_[:2].sum():.1%}")
fg.pca_varianza(pca_dig, marcar=0.9, titulo="Dígitos: 64 componentes",
                leyenda="los dos primeros componentes no llegan a un tercio de la varianza: un dibujo en 2D será borroso. "
                        "Pero bastan 21 de 64 columnas para el 90 %: como compresión, PCA sí sirve.");

**28,5 %** con dos componentes y **21 componentes** para el 90 %. Un dibujo en PC1–PC2 muestra menos de un tercio de la varianza: va a ser borroso. Pero 21 columnas en vez de 64 sí es una buena compresión para alimentar otro modelo.

### t-SNE: dibujar vecindades

**t-SNE** (*t-distributed Stochastic Neighbor Embedding*) no busca direcciones: busca acomodar los puntos en 2D de modo que **cada punto quede cerca de los mismos vecinos** que tenía en 64D. No le importa conservar distancias grandes, solo quién es vecino de quién.

Su perilla principal es la **perplexity**: más o menos, **cuántos vecinos** mira cada punto (valores típicos de 5 a 50).

### ✏️ Código clave — t-SNE de los dígitos
*Ya está resuelto en la celda siguiente: lea qué hace antes de ejecutarlo.*

Guarde en `T_dig` las coordenadas 2D que devuelve un `TSNE(n_components, perplexity, random_state)` con 2 componentes, perplexity 30 y `random_state=42`, usando `.fit_transform(X_dig)`. Tarda unos segundos: mueve las 1 797 posiciones a la vez, en cientos de pasos. Ojo: t-SNE **no tiene `.transform()`**: solo dibuja las filas que recibió; una fila nueva obliga a recalcular todo.

In [ ]:
from sklearn.manifold import TSNE

T_dig = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_dig)
fg.pca_vs_tsne(X_dig, y_dig, pca_dig.transform(X_dig)[:, :2], T_dig);

### 📘 Para entender — t-SNE: para qué sirve y cómo usarlo bien

**Qué es.** Un método solo para **visualizar**: acomoda los puntos en 2D buscando que cada punto conserve a sus vecinos cercanos de las muchas dimensiones originales.

**Cómo usarlo bien:**

| Recomendación | Por qué |
|---|---|
| Pruebe varias `perplexity` (por ejemplo 5, 30, 50) | cada una muestra otra escala de vecindad; una estructura real aparece en todas |
| Escale antes, si las unidades difieren | usa distancias |
| Con muchas columnas, aplique PCA primero (a ~50 componentes) | más rápido y con menos ruido |
| Fije `random_state` y compare varias semillas | el resultado depende del arranque |
| Coloree con algo que **no** se usó para dibujar | es la única forma de saber si los grupos significan algo |
| No reporte tamaños de grupos ni distancias entre grupos | t-SNE los deforma (lo verá enseguida) |
| No lo use como entrada de otro modelo | no tiene `.transform()` y sus ejes no significan nada |

**Dónde se usa:** explorar imágenes, textos y otros datos con cientos de variables; en biología, para ver tipos de células en datos genéticos; para revisar qué "aprendió" una red neuronal mirando cómo agrupa los ejemplos.

**Su pariente moderno: UMAP** (paquete `umap-learn`). La misma idea, más rápido con muchos datos, conserva un poco mejor la estructura global y sí puede ubicar filas nuevas. Tiene los mismos cuidados de interpretación.

Espectacular: t-SNE separa casi todos los dígitos sin haber visto las etiquetas. ¿Entonces t-SNE "encontró" 10 grupos?

<details><summary><b>Para quien quiera más:</b> qué minimiza t-SNE</summary>

Para cada par de puntos calcula una probabilidad de "ser vecinos" en 64D, $p_{ij}$ (una campana gaussiana cuyo ancho se ajusta a la perplexity), y otra en 2D, $q_{ij}$ (una distribución *t* de Student, de colas largas). Mueve los puntos 2D con **descenso de gradiente** (Sesión 4) para minimizar la divergencia de Kullback–Leibler:

$$\text{KL} = \sum_{i \ne j} p_{ij} \log \frac{p_{ij}}{q_{ij}}$$

Esa fórmula castiga mucho separar a dos vecinos verdaderos y casi nada juntar a dos puntos lejanos. Por eso conserva vecindades y no distancias. Y la función tiene muchos mínimos locales (como K-means): el dibujo depende del punto de partida. scikit-learn arranca por defecto desde PCA (`init='pca'`), que lo estabiliza; con el arranque clásico al azar (`init='random'`), cada semilla da otro dibujo.
</details>

### 💥 ¿Le creemos al dibujo?

### 🔍 Predice
Si cambiamos la perplexity a **2** o a **100**, ¿el dibujo se parecerá al anterior?

**Su predicción:** ____

Para que corra rápido, usamos 900 de los 1 797 dígitos.

In [ ]:
muestra = np.random.default_rng(0).choice(len(X_dig), 900, replace=False)
fg.tsne_perplexity(X_dig[muestra], y_dig[muestra], perplexities=(2, 5, 30, 100));

Con perplexity 2 aparecen islotes y astillas que **no existen**; con 100 los grupos se acercan. Ahora la semilla, con el arranque clásico al azar (`init='random'`):

In [ ]:
fg.tsne_semillas(X_dig[muestra], y_dig[muestra], semillas=(0, 1, 2, 3));

Con cada semilla, el mismo dígito cae en otro lugar y con otros vecinos. Lo que sí se repite: **qué** grupos hay. Lo que no: **dónde** quedan y **qué tan lejos** están. (scikit-learn arranca por defecto desde PCA, y así las semillas dan casi el mismo dibujo: más estable, pero no más verdadero; la perplexity lo sigue cambiando todo.)

### 🔍 Predice
Ahora otros datos: 500 puntos tomados al azar, uniformes, en un cuadrado. ¿Qué dibujará t-SNE?

**Su predicción:** ____

In [ ]:
fg.tsne_ruido(n=500, perplexities=(2, 5, 50));

Y una última prueba: tres grupos en 10 dimensiones. **A** es disperso, **B** es diez veces más apretado y está pegado a A, **C** está muy lejos. PCA los muestra como son; ¿y t-SNE?

In [ ]:
fg.tsne_tamanos();

![Cómo leer un t-SNE](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S08/img/como_leer_un_tsne.png)

<details><summary><b>💥 ¿Por qué pasó esto?</b> (ábralo después de la discusión)</summary>

1. **t-SNE dibuja grumos en el ruido.** Con perplexity baja, puntos al azar se ven como decenas de grupitos. Un t-SNE con "grupos" **no prueba** que haya grupos.
2. **Los tamaños no significan nada.** B era diez veces más apretado que A; en t-SNE quedan del mismo tamaño, porque t-SNE "infla" lo denso y "comprime" lo disperso.
3. **Las distancias entre grupos no significan nada.** C estaba lejísimos; en t-SNE queda a la misma distancia que A de B.
4. **Cada perplexity (y, con arranque al azar, cada semilla) da otro dibujo.** No existe "el" t-SNE de unos datos.

Es la mala interpretación de clusters más común en informes y artículos: *"el t-SNE revela tres segmentos de clientes bien separados, y el segmento 2 es el más homogéneo"*. Las tres afirmaciones pueden ser artefactos del dibujo.
</details>

### Entonces, ¿PCA o t-SNE?

| | PCA | t-SNE |
|---|---|---|
| Conserva | la varianza global (distancias grandes) | las vecindades (quién está cerca de quién) |
| Ejes | interpretables (biplot: pesos de cada variable) | sin significado |
| Filas nuevas | sí: `.transform()` | no: no tiene `.transform()`; hay que recalcular todo |
| Repetible | siempre da lo mismo | depende de la perplexity (y de la semilla si arranca al azar) |
| Velocidad | instantáneo | lento con muchas filas |
| Úselo para | comprimir columnas antes de otro modelo; entender qué variables van juntas; un primer dibujo honesto | explorar si hay estructura local; **nunca** como prueba única de grupos |

Regla práctica: **primero PCA**, y si se ve borroso, t-SNE con varias perplexities y semillas, comparando siempre con algo que no se usó para dibujar.

## 8. Todo junto: 1 122 municipios de Colombia

Ahora el flujo completo con datos reales, sin una "verdad de laboratorio": **escalar → PCA para entender → elegir `k` → K-means → perfil → mapa → lo que no se puede concluir**. Es exactamente lo que haría en su proyecto con su propio dataset.

### Los datos: 15 formas de ser pobre

El **IPM** (índice de pobreza multidimensional) no mide ingresos: mide **privaciones**. Un hogar es pobre multidimensional si sufre al menos un tercio (ponderado) de estas 15:

| Dimensión | Privaciones (columna) |
|---|---|
| Educación | `analfabetismo`, `bajo_logro_educativo` |
| Niñez y juventud | `inasistencia_escolar`, `rezago_escolar`, `barreras_primera_infancia`, `trabajo_infantil` |
| Trabajo | `dependencia_economica`, `trabajo_informal` |
| Salud | `sin_aseguramiento_salud`, `barreras_salud` |
| Vivienda y servicios | `sin_agua_mejorada`, `sin_alcantarillado`, `pisos_inadecuados`, `paredes_inadecuadas`, `hacinamiento` |

Cada número es el **% de personas** del municipio que viven en hogares con esa privación.

In [ ]:
muni = pd.read_csv("municipios_ipm_2018.csv", dtype={"cod_mpio": str, "cod_dpto": str})
privaciones = list(muni.columns[10:])
print(muni.shape)
muni.head()

In [ ]:
fg.mapa_municipios(muni, valor="ipm", titulo="IPM por municipio (Censo 2018)");

### 📘 Para entender — qué es el IPM y por qué sirve para este ejercicio

**Qué mide.** La pobreza **multidimensional** no mira cuánto dinero entra al hogar (eso es la pobreza *monetaria*, otra medida del DANE), sino si el hogar **carece** de cosas básicas: educación, condiciones de la niñez, trabajo, salud y vivienda.

**Cómo se calcula** (método de Alkire y Foster, el que usa Colombia):

1. Se miden las **15 privaciones** de cada hogar (sí/no): por ejemplo, "algún adulto no sabe leer", "no hay alcantarillado".
2. Las **5 dimensiones pesan lo mismo** (20 % cada una) y, dentro de cada dimensión, sus privaciones se reparten ese peso por igual.
3. Un hogar es **pobre multidimensional** si su suma ponderada de privaciones llega al **33 %** o más.
4. El **IPM de un municipio**, como se reporta aquí, es el % de personas que viven en hogares pobres.

**Por qué este dataset es bueno para aprender clustering:**

- Tiene muchas columnas en la **misma unidad** (% de personas), así que el escalado no cambia la historia, pero sigue siendo buena práctica.
- Hay una pregunta real detrás: **¿qué tipos de pobreza hay?** No solo cuánta.
- Trae algo para validar sin *y*: el **mapa**.
- Y trae una trampa natural: el IPM está **hecho de** las mismas 15 columnas (si se mete junto a sus partes, el nivel general vota dos veces: es la trampa de la Sesión 9 completa).

### Entender antes de agrupar: PCA

### 🔍 Predice
Vamos a resumir las 15 privaciones con PCA. PC1 se construye **sin mirar el IPM**. ¿Qué tan correlacionado cree que estará PC1 con el IPM oficial del DANE (de −1 a 1)?

**Su predicción:** ____

### ✏️ Código clave — el tramo completo de PCA
*Ya está resuelto en la celda siguiente: lea qué hace antes de ejecutarlo.*

La celda siguiente hace, en un solo tramo:
1. `Z`: las 15 `privaciones` escaladas (promedio 0, desviación 1).
2. `pca`: un PCA con **todos** los componentes, ajustado con `Z`; dibuje su varianza explicada con `fg.pca_varianza` (marque el 80 %).
3. `scores`: los municipios proyectados en los componentes.
4. Imprima la correlación entre PC1 (la primera columna de `scores`) y `muni["ipm"]`.
5. El biplot de PC1–PC2 con las **7** flechas más largas (`fg.biplot`, argumento `top`).

Los nombres `Z`, `pca` y `scores` los usan las celdas siguientes: no los cambie.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

Z = StandardScaler().fit_transform(muni[privaciones])
pca = PCA().fit(Z)
fg.pca_varianza(pca, marcar=0.8, titulo="15 privaciones: ¿cuántas dimensiones de verdad?",
                leyenda="PC1 sola recoge casi la mitad de la variación de 15 columnas: muchas privaciones van juntas. "
                        "15 columnas no son 15 dimensiones de verdad.")
scores = pca.transform(Z)
print("correlación PC1 – IPM:", round(np.corrcoef(scores[:, 0], muni["ipm"])[0, 1], 3))
fg.biplot(scores, pca.components_, privaciones, var_exp=pca.explained_variance_ratio_, top=7);

PC1 recoge **45 %** de la variación de 15 columnas: muchas privaciones van juntas (donde falta acueducto suele faltar alcantarillado, y hay más analfabetismo).

**0,94.** PCA **reinventó el IPM** sin que nadie se lo pidiera: PC1 es "pobreza en general" (todas las flechas apuntan a la derecha). Lo interesante es **PC2** (14 %): separa municipios donde faltan aseguramiento en salud y servicios de primera infancia (arriba) de municipios con bajo logro educativo, trabajo informal y analfabetismo (abajo). Son **tipos** de pobreza distintos con el mismo nivel. Eso es lo que buscamos con K-means.

### Agrupar: ¿cuántos grupos?

En la sección 3 vimos codo y silueta. Hoy agregamos una tercera pregunta: **¿los grupos sobreviven si cambio un poco los datos?** Para cada `k`, se reentrena K-means 20 veces con el 80 % de los municipios y se compara con los grupos originales usando el **índice de Rand ajustado (ARI)**: la fracción de parejas de municipios en la que las dos particiones coinciden (juntos en ambas o separados en ambas), corregida para que el azar dé 0 y dos particiones idénticas den 1.

In [ ]:
fg.estabilidad_k(Z, ks=range(2, 8), marcar=4);

**Lectura honesta:** la silueta es **baja** para todo `k` (≈ 0,2, salvo 0,33 con k = 2): los municipios **no** forman islas separadas; forman un continuo con zonas. Pero la estabilidad es **alta** (0,73 a 0,96 según k): los grupos no son ruido, se repiten.

**¿Por qué k = 4, si no es el más estable?** Con estabilidad 0,82, k = 4 está entre los **menos** estables (k = 6 llega a 0,96). Lo elegimos por **utilidad**: cuatro tipos responden a la pregunta (¿cuatro programas?) y se pueden describir; k = 2 sería más "limpio" pero solo diría "pobres y no pobres", que ya lo dice el IPM, y k = 6 da grupos más finos que nadie pidió. La estabilidad no elige `k`: descarta los que no se sostienen. Y una estabilidad de 0,82 avisa que algunos municipios de frontera cambian de grupo.

### 📘 Para entender — estabilidad y el índice de Rand ajustado (ARI)

**La idea.** Si los grupos son reales, deberían **repetirse** cuando los datos cambian un poco. Se reentrena el clustering muchas veces con muestras del 80 % de las filas y se compara cada resultado con el original.

**El ARI** compara dos particiones de las mismas filas sin importar cómo se llamen los grupos (el "grupo 2" de una corrida puede ser el "grupo 0" de otra). Mira todas las parejas de filas y cuenta en cuántas las dos particiones están de acuerdo (juntas en ambas o separadas en ambas), corrigiendo por lo que daría el azar:

| ARI | Lectura |
|---|---|
| 1 | las dos particiones son idénticas |
| 0,8 – 1 | muy parecidas: los grupos son estables |
| 0,5 – 0,8 | se parecen, pero hay muchas filas de frontera que cambian |
| ≈ 0 | no se parecen más que por azar |

```python
from sklearn.metrics import adjusted_rand_score
adjusted_rand_score(grupos_corrida_1, grupos_corrida_2)
```

El ARI también sirve para comparar un clustering con una **etiqueta real** (especies de pingüinos, región, tipo de cliente): es una forma de validación externa con un solo número.

**Cómo elegir $k$, en resumen:** use las tres preguntas juntas y en este orden de importancia: (1) ¿sirve para la pregunta? (cuántos programas, cuántas campañas); (2) ¿se sostiene? (estabilidad); (3) ¿qué dice la geometría? (codo y silueta). Una silueta baja con estabilidad alta, como hoy, describe un **continuo con zonas**: los grupos son una forma útil de partirlo, no islas naturales.

### ✏️ Código clave — K-means con k = 4
*Ya está resuelto en la celda siguiente: lea qué hace antes de ejecutarlo.*

`kmeans`: K-means con 4 grupos sobre `Z`, 10 arranques y `random_state=42`. Los números de cluster son nombres arbitrarios; para leer más fácil, la celda los **renumera** del menos al más pobre.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Z)
orden = muni["ipm"].groupby(kmeans.labels_).mean().sort_values().index      # del menos al más pobre
muni["grupo"] = pd.Series(kmeans.labels_).map({viejo: nuevo for nuevo, viejo in enumerate(orden)})
muni.groupby("grupo")["ipm"].agg(["count", "mean", "min", "max"]).round(1)

### Leer los grupos: perfil y mapa

### 🔍 Predice
Antes de ver el perfil: ¿cree que los grupos 1 y 2 (IPM medio) se diferencian **solo** en el nivel de pobreza, o en el **tipo**? ¿En qué privación esperaría la mayor diferencia?

**Su predicción:** ____

In [ ]:
fg.perfil_clusters(muni, muni["grupo"], ["ipm"] + privaciones);

### 📘 Para entender — cómo perfilar y nombrar los grupos

Un clustering no termina en las etiquetas: termina cuando cada grupo tiene un **nombre** que alguien pueda usar para decidir.

1. **Tamaño de cada grupo.** Un grupo con 3 filas no es un segmento, es una curiosidad (o un error de datos).
2. **Perfil en unidades originales**: el promedio (o la mediana) de cada variable por grupo, comparado con el promedio general. Lo que **distingue** a un grupo es lo que se aleja del promedio, no lo que es alto en todos.
3. **El nombre** describe lo que lo distingue: "rural andino con bajo logro educativo y vivienda aceptable" dice más que "grupo 1".
4. **Validación externa**: ¿coincide con algo que el algoritmo no vio? (hoy, la región y el mapa).
5. **De grupo a acción**: ¿qué haría distinto con cada grupo?

Ejemplos de segmentaciones que terminan en acción:

| Contexto | Grupos típicos | Decisión distinta por grupo |
|---|---|---|
| Política pública (hoy) | urbano · rural andino · Caribe con déficit de vivienda · periferia | educación de adultos · alcantarillado y vivienda · acceso a agua y salud |
| Clientes de un banco | ahorradores · usuarios de crédito · inactivos | ofrecer inversión · ofrecer refinanciación · campaña de reactivación |
| Tiendas de una cadena | alto tráfico y ticket bajo · bajo tráfico y ticket alto | surtido de conveniencia · surtido premium |

In [ ]:
fg.mapa_municipios(muni, etiquetas=muni["grupo"], titulo="Cuatro tipos de municipio según sus privaciones");

In [ ]:
# Validación externa: la región NO se usó para agrupar.
# Región asignada por departamento: una aproximación gruesa (Antioquia o Nariño, por ejemplo, tienen municipios de dos regiones).
region = muni["cod_dpto"].map({"05": "Andina", "08": "Caribe", "11": "Andina", "13": "Caribe", "15": "Andina", "17": "Andina",
    "18": "Amazonía", "19": "Pacífica", "20": "Caribe", "23": "Caribe", "25": "Andina", "27": "Pacífica", "41": "Andina",
    "44": "Caribe", "47": "Caribe", "50": "Orinoquía", "52": "Pacífica", "54": "Andina", "63": "Andina", "66": "Andina",
    "68": "Andina", "70": "Caribe", "73": "Andina", "76": "Pacífica", "81": "Orinoquía", "85": "Orinoquía", "86": "Amazonía",
    "88": "Caribe", "91": "Amazonía", "94": "Amazonía", "95": "Amazonía", "97": "Amazonía", "99": "Orinoquía"})
pd.crosstab(region.rename("región"), muni["grupo"])

**Lectura:**

| Grupo | Municipios | IPM medio | Qué lo distingue | Dónde |
|---|---|---|---|---|
| 0 | 286 | 22 | menos privaciones en todo; menos informalidad (79 %) | ciudades y sus alrededores (Bogotá, Medellín, Cali, Barranquilla, Pasto) |
| 1 | 474 | 40 | bajo logro educativo alto (73 %, casi como el grupo 2) pero vivienda y servicios aceptables (sin alcantarillado 17 %, pisos 14 %) | la **zona rural andina** (Boyacá, Santander, Cundinamarca, Antioquia) |
| 2 | 309 | 56 | **vivienda y servicios**: sin alcantarillado (42 %), pisos inadecuados (28 %), hacinamiento (17 %) | el **Caribe** (166 de los 197 municipios caribeños) |
| 3 | 53 (1 sin coordenadas: no está en el mapa) | 83 | todo en rojo; sin agua mejorada (78 %), sin aseguramiento (35 %), barreras de primera infancia | **Amazonía, Chocó, La Guajira**: periferia |

Los grupos 1 y 2 **comparten** el bajo logro educativo (73 % y 69 %): lo que los separa es la **vivienda y los servicios** (sin alcantarillado 17 % contra 42 %, pisos 14 % contra 28 %, hacinamiento 8 % contra 17 %). Y su IPM se traslapa (el grupo 1 llega a 67 y el 2 baja a 37): no son solo "medio-bajo" y "medio-alto". Un programa de educación de adultos les sirve a los dos; uno de vivienda y alcantarillado, sobre todo al 2. Y el mapa confirma que los grupos significan algo, porque K-means **no vio las coordenadas** y aun así dibujó las regiones.

### Lo que se puede y lo que NO se puede concluir

| ✓ Se puede decir | ✗ No se puede decir |
|---|---|
| Hay cuatro tipos de municipio estables según sus privaciones en 2018 | que esos cuatro grupos "existen" en la naturaleza (otro `k` u otras columnas darían otros) |
| El grupo 2 se distingue por vivienda y servicios, y es mayoritariamente caribeño | que ser caribeño **causa** pisos de tierra (el clustering no habla de causas) |
| Quibdó comparte perfil con municipios del Caribe rural | que **cada persona** de un municipio del grupo 3 es pobre (eso es la **falacia ecológica**: confundir el promedio del grupo con sus individuos) |
| Los grupos coinciden con regiones que el algoritmo no vio | que 286 municipios "ricos" son el 25 % de la población: cada municipio cuenta 1, sea Bogotá o Pana Pana |

Y un límite de los datos: **es 2018**. Un censo es una foto; el clustering describe esa foto.

### 📘 Para entender — las trampas de interpretación con datos agregados

| Trampa | Qué es | Ejemplo con estos datos |
|---|---|---|
| **Falacia ecológica** | atribuir a cada individuo lo que vale para el promedio del grupo | "todos los habitantes de un municipio del grupo 3 son pobres" |
| **Unidad de análisis** | los resultados son sobre la unidad de la fila, no sobre otra | cada municipio pesa 1: Bogotá (7 millones de personas) cuenta igual que un municipio de 2 000 |
| **Ponderación** | cambiar el peso de cada fila cambia los promedios | para hablar de *personas* habría que ponderar por población |
| **Causalidad** | que dos cosas vayan juntas no dice cuál causa cuál | "ser del Caribe causa déficit de vivienda" no se sigue del clustering |
| **Foto en el tiempo** | los datos describen un momento | el censo es de 2018; la realidad pudo cambiar |

Una variante famosa es la **paradoja de Simpson**: una relación que se ve en los datos agregados puede invertirse dentro de cada subgrupo. Antes de concluir algo sobre personas con datos de municipios (o sobre clientes con datos de tiendas), pregunte siempre: *¿de qué es cada fila?*

## 9. Los algoritmos en una ficha (resumen flash para repasar)

Cuatro algoritmos, la misma plantilla: **qué hace**, **qué pasa dentro del `.fit()`**, **sus perillas**, **cuándo sí y cuándo no**, **su trampa** y **el código mínimo**. Las cifras son las de este notebook.

### 🟠 K-means — agrupar alrededor de centros

| | |
|---|---|
| **En una frase** | parte las filas en $k$ grupos; cada grupo es una nube alrededor de su **centroide** (el promedio de sus puntos) |
| **Dentro del `.fit()`** | repite *asignar* (cada punto al centroide más cercano) y *promediar* (cada centroide al promedio de sus puntos) hasta que nada se mueve (algoritmo de Lloyd) |
| **Qué minimiza** | la **inercia**: $\sum_i \lVert x_i - \mu_{c(i)} \rVert^2$, la suma de distancias² de cada punto a su centroide. Llega a un **mínimo local**: por eso `n_init` |
| **Perillas** | `n_clusters` ($k$, lo decide usted) · `n_init` (10 arranques) · `random_state` |
| **¿Escalar?** | **sí, siempre** (pingüinos: 68 % sin escalar → 92 % escalado) |
| **¿Filas nuevas?** | sí: `.predict()` las manda al centroide más cercano |
| **Úselo para** | segmentar clientes, tiendas o municipios en grupos compactos que se describen con un "prototipo" |
| **No lo use para** | grupos con formas raras (lunas, anillos) o con mucho ruido: siempre fuerza cada punto en un grupo |
| **Su trampa** | siempre entrega $k$ grupos, existan o no; y sin escalar agrupa por la unidad más grande |

```python
km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Z)   # Z = datos escalados, SIN y
km.labels_, km.cluster_centers_, km.inertia_
```

### 🟢 DBSCAN — agrupar por densidad

| | |
|---|---|
| **En una frase** | un grupo es una **zona densa** de puntos, con la forma que tenga; los puntos aislados son **ruido** (etiqueta −1) |
| **Dentro del `.fit()`** | marca como **núcleo** a cada punto con al menos `min_samples` vecinos a distancia ≤ $\varepsilon$, encadena núcleos vecinos y les pega sus **bordes**. No minimiza nada y no tiene semilla |
| **Perillas** | `eps` ($\varepsilon$: qué tan cerca es cerca) · `min_samples` (cuántos vecinos para ser núcleo; regla práctica: el doble de columnas) |
| **¿Escalar?** | **sí** (`eps` es una distancia en las unidades de los datos) |
| **¿Filas nuevas?** | no tiene `.predict()`: se usa `.fit_predict()` |
| **Úselo para** | formas raras, puntos calientes en mapas, detectar anomalías (lo que queda como ruido) |
| **No lo use para** | grupos de densidades muy distintas (un solo `eps` no sirve para todos: pruebe HDBSCAN) |
| **Su trampa** | `eps` lo cambia todo: lunas con 0,3 → 1 grupo; con 0,15 → 2; con 0,05 → decenas de grupitos y casi todo ruido |

```python
etiquetas = DBSCAN(eps=0.15, min_samples=5).fit_predict(Z)    # −1 = ruido
```

### 🔵 PCA — resumir columnas en pocas direcciones

| | |
|---|---|
| **En una frase** | rota los ejes: PC1 es la dirección donde más se esparcen los datos, PC2 la siguiente (perpendicular), y así |
| **Dentro del `.fit()`** | cada componente es una suma ponderada $w_1x_1 + \dots + w_px_p$ con los pesos que **maximizan la varianza** (con $\sum w_j^2 = 1$). Es álgebra lineal: sin iteraciones ni semilla, siempre da lo mismo |
| **Qué mide** | la **varianza explicada** de cada componente, $\lambda_k / \sum \lambda$ (pingüinos escalados: PC1 69 % + PC2 19 % = 88 %; dígitos: 2 componentes 28,5 %, 21 para el 90 %) |
| **Perillas** | `n_components` (un número, o una fracción como `0.90` para que elija solo) |
| **¿Escalar?** | sí, si las unidades difieren (sin escalar, PC1 = la masa en gramos con 99,99 %) |
| **¿Filas nuevas?** | sí: `.transform()` |
| **Úselo para** | entender qué variables van juntas (biplot), construir índices (PC1 de las privaciones ≈ IPM, r = 0,94), comprimir antes de otro modelo |
| **No lo use para** | relaciones curvas (solo ve lo lineal), o cuando "máxima varianza" no es lo que le importa |
| **Su trampa** | sin escalar, "descubre" la variable con la unidad más grande |

```python
pca = PCA().fit(Z)
pca.explained_variance_ratio_      # cuánto recoge cada componente
pca.components_                    # los pesos: qué significa cada eje
scores = pca.transform(Z)          # las filas en los ejes nuevos
```

### 🟣 t-SNE — dibujar vecindades

| | |
|---|---|
| **En una frase** | acomoda los puntos en 2D para que cada uno quede cerca de **los mismos vecinos** que tenía en muchas dimensiones |
| **Dentro del `.fit()`** | compara probabilidades de "ser vecinos" en el espacio original ($p_{ij}$) y en el dibujo ($q_{ij}$) y mueve los puntos con **descenso de gradiente** para minimizar la divergencia KL. Tiene mínimos locales |
| **Perillas** | `perplexity` (≈ cuántos vecinos mira cada punto, 5–50) · `random_state` |
| **¿Escalar?** | sí, si las unidades difieren |
| **¿Filas nuevas?** | **no**: no tiene `.transform()`; una fila nueva obliga a recalcular todo |
| **Úselo para** | **explorar** si hay estructura local en datos de muchas columnas (imágenes, textos, genes) |
| **No lo use para** | probar que hay grupos, medir tamaños de grupos o distancias entre grupos, ni como entrada de otro modelo |
| **Su trampa** | dibuja grumos en el **ruido puro**, iguala tamaños y distancias, y cambia con la perplexity (y la semilla) |

```python
T = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X)   # pruebe varias perplexities
```

### ¿Cuál uso? Una guía rápida

```
¿Qué quiere?
├── AGRUPAR filas parecidas
│     ├── espera grupos compactos y quiere describirlos con un centro ........ K-means (escale; elija k con utilidad, estabilidad, silueta)
│     ├── espera formas raras, ruido o no sabe cuántos grupos ............... DBSCAN / HDBSCAN (escale; pruebe varios eps)
│     └── quiere ver cómo se van uniendo los grupos (pocos miles de filas) ... jerárquico (dendrograma)
└── RESUMIR muchas columnas
      ├── entender qué variables van juntas, construir un índice, comprimir .. PCA (escale si las unidades difieren)
      └── solo DIBUJAR vecindades para explorar ............................. t-SNE / UMAP (varias perplexities; nunca como prueba)
```

### Cómo se evalúa sin *y* (las cuatro preguntas)

| Pregunta | Herramienta | En este notebook |
|---|---|---|
| ¿Qué dice la geometría? | **inercia** (codo) y **silueta** (−1 a 1, más alto = grupos más separados) | blobs: silueta máxima en k = 3 (0,63), aunque fabricamos 4 |
| ¿Se sostiene si cambio un poco los datos? | **estabilidad** con el índice de Rand ajustado (**ARI**: 1 = idénticos, 0 = azar) | municipios: ARI 0,73–0,96 según k |
| ¿Coincide con algo que el algoritmo no vio? | **validación externa**: tabla cruzada, mapa, ARI contra una etiqueta | especie de los pingüinos; región y mapa de los municipios |
| ¿Sirve para decidir algo distinto en cada grupo? | **utilidad**: nombres y acciones por grupo | cuatro tipos de municipio, cuatro programas |

### Dentro del `.fit()` — los cuatro, lado a lado

| | K-means | DBSCAN | PCA | t-SNE |
|---|---|---|---|---|
| Tarea | agrupar | agrupar (densidad) | reducir dimensiones | dibujar vecindades |
| Qué optimiza | minimiza inercia | nada (encadena vecinos) | maximiza varianza proyectada | minimiza KL entre vecindades |
| Cómo lo busca | iterativo (Lloyd), mínimo local | regla de vecindad | álgebra lineal, solución exacta | descenso de gradiente, mínimo local |
| ¿Semilla? | sí (`n_init`) | no | no | sí (menos con `init='pca'`, el valor por defecto) |
| Perilla clave | `n_clusters` | `eps`, `min_samples` | `n_components` | `perplexity` |
| ¿Filas nuevas? | `.predict()` | no | `.transform()` | no |
| ¿Escalar? | sí | sí | si las unidades difieren | si las unidades difieren |

## 10. ⭐ Opcional para su proyecto: un análisis no supervisado

La entrega final del proyecto (Sesiones 10–11) es supervisada. Si quiere **sumar puntos**, agregue una sección no supervisada sobre su mismo dataset, **sin usar la *y*** (la deja guardada para leer los grupos al final):

1. **Columnas:** solo numéricas, sin la *y*, sin identificadores y sin resúmenes de otras columnas.
2. **Escalar** con `StandardScaler`.
3. **PCA** para entender: varianza explicada, biplot y qué significa PC1.
4. **K-means** (o DBSCAN), con `k` justificado por utilidad, estabilidad y silueta.
5. **Perfil y nombre** de cada grupo.
6. **Validación externa**: compare los grupos con su *y* o con una categórica que no usó.
7. **Tres frases de lo que no se puede concluir** (causalidad, estabilidad, individuos contra grupos).

Plantilla mínima (cambie lo que está en MAYÚSCULAS). Si su dataset tiene más de 5 000 filas, use una **muestra** de 5 000: la estabilidad y la silueta reentrenan K-means muchas veces.

In [ ]:
# ---------- PLANTILLA E3 ----------
# df = pd.read_csv("/content/drive/MyDrive/SU_CARPETA/SUS_DATOS.csv")
# columnas = ["COL_1", "COL_2", "COL_3"]                # numéricas, sin la y, sin identificadores, sin categóricas
# df = df.dropna(subset=columnas)
# if len(df) > 5000:
#     df = df.sample(5000, random_state=42)              # para que estabilidad y silueta corran en Colab
# df = df.reset_index(drop=True)
# Z = StandardScaler().fit_transform(df[columnas])
#
# pca = PCA().fit(Z);  fg.pca_varianza(pca)
# fg.biplot(pca.transform(Z), pca.components_, columnas, var_exp=pca.explained_variance_ratio_)
#
# fg.estabilidad_k(Z, ks=range(2, 8))
# km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(Z)
# fg.perfil_clusters(df, km.labels_, columnas)
# pd.crosstab(df["SU_Y"], km.labels_)                   # validación externa: la y o una categórica que no usó

## 11. Cierre — el Módulo 3 en tres frases

![El flujo no supervisado](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S09/img/flujo_no_supervisado.png)

1. **Sin *y*, nadie delata el error.** Un algoritmo de clustering siempre devuelve grupos y un t-SNE siempre dibuja algo: cada decisión (columnas, escala, `k`, perplexity) se justifica y cada conclusión se valida con algo externo.
2. **Las unidades deciden si no las controla.** Sin escalar, K-means agrupó pingüinos por gramos y PCA "descubrió" la masa corporal.
3. **Un resultado que cambia con la semilla, la perplexity o las columnas no es un hallazgo.** Lo que se repite y coincide con algo que el algoritmo no vio (la especie, el mapa) sí lo es.

| Trampa del módulo | Decisión del analista | Qué pasó |
|---|---|---|
| 💥 Agrupó por gramos (sección 5) | unidades de medida | K-means formó franjas de peso: 68 % → 92 % al escalar |
| 💥 ¿Le creemos al dibujo? (sección 7) | el dibujo (perplexity, semilla) | t-SNE inventó grupos en ruido puro e igualó tamaños y distancias |
| 💥 El IPM vota dos veces (Sesión 9 completa) | qué columnas entran | meter un resumen junto a sus partes ordena los grupos por nivel |

**Lo que sigue:** presentaciones del proyecto final y cierre del curso.

### 📘 El Módulo 3 en una página (para el proyecto y para repasar)

| Paso | Qué hacer | Herramienta | Trampa que evita |
|---|---|---|---|
| 1. Columnas | elegir las que describen lo que quiere agrupar; sin *y*, sin identificadores, sin resúmenes de otras columnas | criterio + `df.corr()` | votar dos veces (Sesión 9) |
| 2. Escalar | poner las variables en unidades comparables | `StandardScaler` | agrupar por unidades (sección 5) |
| 3. Entender | ver cuántas dimensiones reales hay y qué significan | `PCA`, biplot | — |
| 4. Agrupar | elegir algoritmo y perillas | `KMeans`, `DBSCAN` | grupos forzados |
| 5. Elegir $k$ | utilidad, estabilidad, geometría | ARI, silueta, codo | creerle a una sola métrica |
| 6. Perfilar | promedio por grupo en unidades originales; nombres | `groupby`, `fg.perfil_clusters` | grupos sin significado |
| 7. Validar | comparar con algo que no se usó | tabla cruzada, mapa, ARI | grupos que solo existen en el algoritmo |
| 8. Límites | decir qué **no** se puede concluir | — | falacia ecológica, causalidad |
| (Dibujar) | si usa t-SNE, varias perplexities y semillas | `TSNE` | creerle al dibujo (sección 7) |